# MBIR and Pyramid Magnetic-Field Reconstruction

This notebook demonstrates the **Model-Based Iterative Reconstruction (MBIR)** functionality in `libertem_holo`, which uses [`unxt`](https://github.com/GalileoAstronomia/unxt) to carry physical units through the entire computation.

All input and output quantities that carry physical units are represented as `unxt.Quantity` objects, making the computation explicit:
- Pixel size: `unxt.Quantity(5.0, 'nm')` — nanometres per pixel
- Sample thickness: `unxt.Quantity(50.0, 'nm')` — projected thickness in nanometres
- Reconstructed B-field: output in Tesla `'T'` or millitesla `'mT'`

## Contents
1. Synthetic divergence-free B-field test
2. Simple gradient inversion (`phase_to_b_field`)
3. Tikhonov-regularized MBIR (`reconstruct_b_field_tikhonov`)
4. L-curve for regularization-parameter selection
5. Multi-scale pyramid reconstruction (`reconstruct_b_field_pyramid`)
6. Comparison of methods

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import unxt

from libertem_holo.base.mbir import (
    b_field_to_phase,
    compute_lcurve,
    phase_scale_factor,
    phase_to_b_field,
    plot_b_field,
    plot_lcurve,
    reconstruct_b_field_pyramid,
    reconstruct_b_field_tikhonov,
    ELEMENTARY_CHARGE,
    HBAR,
    FLUX_QUANTUM,
)

## 1 · Physical constants and unit system

The module exposes all relevant physical constants as `unxt.Quantity` objects.

In [ ]:
print(f"Elementary charge:  {ELEMENTARY_CHARGE}")
print(f"Reduced Planck:     {HBAR}")
print(f"Flux quantum Φ₀:    {FLUX_QUANTUM}")

## 2 · Experiment parameters

Set up the microscope and sample parameters as `unxt.Quantity` objects.
Changing these propagates automatically through all subsequent calculations.

In [ ]:
# Microscope / reconstruction parameters
voxel_size = unxt.Quantity(3.0, 'nm')   # real-space pixel size in nm
thickness  = unxt.Quantity(40.0, 'nm')  # projected sample thickness in nm

# Derived conversion factor  C = e·t·Δx / ħ  [rad / T]
C = phase_scale_factor(voxel_size=voxel_size, thickness=thickness)
print(f"Phase scale factor C = {C}")
print(f"  (a B-field of 1 T produces a phase gradient of {float(C.value):.4f} rad/pixel)")

## 3 · Synthetic phase image

Create a synthetic magnetic sample: a pair of magnetic vortices
(divergence-free field ⇒ representable by a scalar magnetic phase).

In [ ]:
rng = np.random.default_rng(42)

ny, nx = 128, 128
y, x = np.mgrid[-ny//2:ny//2, -nx//2:nx//2]

sigma = 12.0

# Two Gaussian vector-potential blobs → divergence-free B field
Az1 = np.exp(-((x - 20)**2 + (y - 10)**2) / (2 * sigma**2))
Az2 = -np.exp(-((x + 20)**2 + (y - 10)**2) / (2 * sigma**2))
Az  = Az1 + Az2

dy_Az, dx_Az = np.gradient(Az)
bx_true = dy_Az   # Bx = ∂Az/∂y
by_true = -dx_Az  # By = −∂Az/∂x

# Wrap in unxt.Quantity (values in Tesla)
bx_q = unxt.Quantity(bx_true, 'T')
by_q = unxt.Quantity(by_true, 'T')

print(f"|B| max: {np.sqrt(bx_true**2 + by_true**2).max():.4f} T")

# Forward model: compute expected magnetic phase
phase_clean = b_field_to_phase(
    bx_q, by_q,
    voxel_size=voxel_size,
    thickness=thickness,
)
print(f"Phase range (clean): [{np.asarray(phase_clean.value).min():.3f},"
      f" {np.asarray(phase_clean.value).max():.3f}] {phase_clean.unit}")

# Add Gaussian noise to simulate experimental conditions
noise_level = 0.05  # rad (root-mean-square)
noise = rng.standard_normal((ny, nx)) * noise_level
phase_noisy = unxt.Quantity(
    np.asarray(phase_clean.value) + noise, 'rad'
)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

axes[0].imshow(np.sqrt(bx_true**2 + by_true**2), cmap='viridis', origin='lower')
axes[0].set_title('|B| true [T]')
plt.colorbar(axes[0].images[0], ax=axes[0])

axes[1].imshow(np.asarray(phase_clean.value), cmap='RdBu_r', origin='lower')
axes[1].set_title(f'Phase clean [{phase_clean.unit}]')
plt.colorbar(axes[1].images[0], ax=axes[1])

axes[2].imshow(np.asarray(phase_noisy.value), cmap='RdBu_r', origin='lower')
axes[2].set_title(f'Phase with noise [{phase_noisy.unit}]')
plt.colorbar(axes[2].images[0], ax=axes[2])

for ax in axes:
    pix_nm = float(unxt.ustrip('nm', voxel_size))
    ax.set_xlabel(f'x [pixel × {pix_nm:.1f} nm]')
    ax.set_ylabel(f'y [pixel × {pix_nm:.1f} nm]')

plt.tight_layout()
plt.show()

## 4 · Simple gradient inversion

The simplest inversion: compute B directly from the phase gradient.
This is equivalent to Tikhonov with λ → 0 but amplifies high-frequency noise.

In [ ]:
bx_grad, by_grad = phase_to_b_field(
    phase_noisy,
    voxel_size=voxel_size,
    thickness=thickness,
    output_unit='mT',
)
print(f"Gradient method output unit: {bx_grad.unit}")
print(f"|Bx| max: {np.abs(np.asarray(bx_grad.value)).max():.2f} {bx_grad.unit}")

## 5 · L-curve for regularization parameter selection

Run the L-curve analysis to select the optimal Tikhonov regularization parameter λ.
The corner of the L-curve balances data fidelity against solution smoothness.

In [ ]:
lcurve = compute_lcurve(
    phase_noisy,
    voxel_size=voxel_size,
    thickness=thickness,
    n_lambdas=40,
    lambda_min=1e-8,
    lambda_max=1e4,
)
print(f"Optimal λ from L-curve: {lcurve.optimal_lambda:.2e}")

In [ ]:
plot_lcurve(lcurve)
plt.show()

## 6 · Tikhonov MBIR reconstruction

Perform regularised reconstruction using the optimal λ. 
The result carries `unxt.Quantity` units throughout.

In [ ]:
lam_opt = lcurve.optimal_lambda

result_tik = reconstruct_b_field_tikhonov(
    phase_noisy,
    voxel_size=voxel_size,
    thickness=thickness,
    regularization_parameter=lam_opt,
    output_unit='mT',
)

print(f"Bx unit: {result_tik.b_x.unit}")
print(f"By unit: {result_tik.b_y.unit}")
print(f"|B| unit: {result_tik.b_magnitude.unit}")
print(f"Voxel size: {result_tik.voxel_size}")
print(f"λ used: {result_tik.regularization_parameter:.2e}")
print(f"|B| max: {np.asarray(result_tik.b_magnitude.value).max():.2f} {result_tik.b_magnitude.unit}")

In [ ]:
plot_b_field(result_tik)
plt.suptitle(f'Tikhonov MBIR  (λ = {lam_opt:.2e})', y=1.01)
plt.tight_layout()
plt.show()

## 7 · Pyramid (multi-scale) reconstruction

The pyramid method applies Tikhonov reconstruction at multiple spatial scales,
progressively refining the estimate from coarse to fine resolution.

In [ ]:
result_pyr = reconstruct_b_field_pyramid(
    phase_noisy,
    voxel_size=voxel_size,
    thickness=thickness,
    regularization_parameter=lam_opt,
    n_levels=3,
    output_unit='mT',
)

print(f"Pyramid Bx unit: {result_pyr.b_x.unit}")
print(f"|B| max: {np.asarray(result_pyr.b_magnitude.value).max():.2f} {result_pyr.b_magnitude.unit}")

In [ ]:
plot_b_field(result_pyr)
plt.suptitle(f'Pyramid MBIR  (λ = {lam_opt:.2e}, 3 levels)', y=1.01)
plt.tight_layout()
plt.show()

## 8 · Comparison: gradient vs Tikhonov vs Pyramid

In [ ]:
fig, axes = plt.subplots(3, 3, figsize=(13, 12))

# True field (scale to mT for comparison)
bmag_true = np.sqrt(bx_true**2 + by_true**2) * 1000  # T → mT
vmax = np.percentile(bmag_true, 99)

# Row 0: True
axes[0, 0].imshow(bx_true * 1000, cmap='RdBu_r', vmin=-vmax, vmax=vmax, origin='lower')
axes[0, 0].set_title('$B_x$ true [mT]')
plt.colorbar(axes[0, 0].images[0], ax=axes[0, 0])
axes[0, 1].imshow(by_true * 1000, cmap='RdBu_r', vmin=-vmax, vmax=vmax, origin='lower')
axes[0, 1].set_title('$B_y$ true [mT]')
plt.colorbar(axes[0, 1].images[0], ax=axes[0, 1])
axes[0, 2].imshow(bmag_true, cmap='viridis', vmin=0, vmax=vmax, origin='lower')
axes[0, 2].set_title('|B| true [mT]')
plt.colorbar(axes[0, 2].images[0], ax=axes[0, 2])

# Row 1: Gradient method (convert bx_grad to mT already)
bx_g = np.asarray(bx_grad.value)
by_g = np.asarray(by_grad.value)
bmag_g = np.sqrt(bx_g**2 + by_g**2)
axes[1, 0].imshow(bx_g, cmap='RdBu_r', vmin=-vmax, vmax=vmax, origin='lower')
axes[1, 0].set_title(f'$B_x$ gradient [{bx_grad.unit}]')
plt.colorbar(axes[1, 0].images[0], ax=axes[1, 0])
axes[1, 1].imshow(by_g, cmap='RdBu_r', vmin=-vmax, vmax=vmax, origin='lower')
axes[1, 1].set_title(f'$B_y$ gradient [{bx_grad.unit}]')
plt.colorbar(axes[1, 1].images[0], ax=axes[1, 1])
axes[1, 2].imshow(bmag_g, cmap='viridis', vmin=0, vmax=vmax, origin='lower')
axes[1, 2].set_title(f'|B| gradient [{bx_grad.unit}]')
plt.colorbar(axes[1, 2].images[0], ax=axes[1, 2])

# Row 2: Pyramid method
bx_p = np.asarray(result_pyr.b_x.value)
by_p = np.asarray(result_pyr.b_y.value)
bmag_p = np.asarray(result_pyr.b_magnitude.value)
axes[2, 0].imshow(bx_p, cmap='RdBu_r', vmin=-vmax, vmax=vmax, origin='lower')
axes[2, 0].set_title(f'$B_x$ pyramid [{result_pyr.b_x.unit}]')
plt.colorbar(axes[2, 0].images[0], ax=axes[2, 0])
axes[2, 1].imshow(by_p, cmap='RdBu_r', vmin=-vmax, vmax=vmax, origin='lower')
axes[2, 1].set_title(f'$B_y$ pyramid [{result_pyr.b_y.unit}]')
plt.colorbar(axes[2, 1].images[0], ax=axes[2, 1])
axes[2, 2].imshow(bmag_p, cmap='viridis', vmin=0, vmax=vmax, origin='lower')
axes[2, 2].set_title(f'|B| pyramid [{result_pyr.b_magnitude.unit}]')
plt.colorbar(axes[2, 2].images[0], ax=axes[2, 2])

row_labels = ['True', 'Gradient', 'Pyramid']
for ax, label in zip(axes[:, 0], row_labels):
    ax.set_ylabel(label, fontsize=13, fontweight='bold')

pix_nm = float(unxt.ustrip('nm', voxel_size))
for ax in axes.ravel():
    ax.set_xlabel(f'x [{pix_nm:.1f} nm/px]')

plt.tight_layout()
plt.show()

## 9 · Summary

This notebook demonstrated the MBIR magnetic-field reconstruction using `unxt` for explicit unit tracking:

| Parameter | Value |
|-----------|-------|
| Pixel size | `{voxel_size}` |
| Thickness  | `{thickness}` |
| Phase-to-B factor C | `{C}` |
| Optimal λ (L-curve) | `{lam_opt:.2e}` |
| Output unit | `mT` |

Key takeaways:
- **`unxt.Quantity`** ensures all physical inputs (voxel size, thickness) carry their units, preventing unit errors.
- All **output B-field quantities** are `unxt.Quantity` objects with explicit Tesla or millitesla units.
- The **L-curve** provides a principled way to select the regularization parameter λ without ground truth.
- The **pyramid method** gives smoother reconstructions than single-scale Tikhonov by combining multiple resolution levels.